In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver

In [2]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [4]:
from dotenv import load_dotenv
load_dotenv()
llm = model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [6]:
def chat_node(state: ChatState) -> ChatState:
    messages = state['messages']
    response = llm.invoke(messages)
    return{'messages': [response]}

In [20]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)
workflow = graph.compile(checkpointer=checkpointer)

In [11]:
initial_state = {'messages': [HumanMessage(content="What is the capital of Zimbawe?")]}
llm.invoke(initial_state['messages']).content

'The capital of Zimbabwe is Harare.'

In [21]:
thread_id = '1'
while True:

    user_message = input("Type your message ....")

    print(f"User: {user_message}")

    if user_message.strip().lower() in ["exit", "quit"]:
        print("Exiting the chat.")
        break

    config = {'configurable': {'thread_id': thread_id}}

    result = workflow.invoke({"messages": [HumanMessage(content=user_message)]}, config=config)
    

    print(result["messages"][-1].content)

User: my name is gudu
Hello Gudu, it's nice to meet you. Is there something I can help you with or would you like to chat?
User: what is name
A "name" is a word or phrase that identifies a person, place, thing, or idea. It's a label or a designation that distinguishes one thing from another.

In your case, "Gudu" is your name, which is a unique identifier that refers to you as an individual. Names can be given to people, animals, objects, locations, and even concepts, and they help us to communicate and distinguish between different things.

Names can have different types, such as:

* Given name (the name given to a person at birth)
* Surname (a family name)
* Nickname (an informal name used by friends or family)
* Place name (the name of a location)
* Brand name (the name of a product or company)

What do you think about names, Gudu? Do you like your name, or is there a story behind how you got it?
User: what is my name then
I remember, your name is **Gudu**. You told me that earlier.